In [ ]:
#| default_exp card

In [ ]:
#| export
from __future__ import annotations

import re

from fastermodels.eval import wilson

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

The reference is named on the card, so `meta['reference']` carries a `name`, the `k`/`n` it was measured on,
and the same criteria as a row (`bytes`, `params`, `macs`, `peak_activation_bytes`); without the first three
`render_card` raises rather than compare to something anonymous.

A row may also carry a `target`: the accuracy the variant aimed for. It is reported under the table — met
when the lower bound of the paired interval clears it, **not demonstrated** otherwise, which is a statement
about the evidence and not a failure: the target is an objective, not a condition to publish, since a more
compressed variant serves a more constrained use. `meta['ladder']` lists the sibling variants of the same
source model so a reader can pick the point that suits them.

The front matter carries a `license:` key only once a person has validated it: with
`license={'id': ..., 'validated_by': ''}` the key is left out and the provenance block says the license is
not yet validated, so the Hub never shows a license nobody checked. `base_model:` follows the same rule for
a different reason — the Hub only accepts one of its own model ids there, so a source named by its factory
(`torchvision.models.resnet18 (IMAGENET1K_V1)`) is written as `Source model` in the provenance block
instead, where it says as much and breaks nothing.

The card carries four criteria and nothing else: **top-1** (with the paired delta against the reference),
**size** (bytes on disk and weights), **memory** (peak live activations for one image) and **MACs**. Each is
shown three times — the reference, this artifact, and the gap between them — so a number is never read
alone. The gap is a signed percentage, never an `N×` ratio: a model that is 74 % smaller is not a model
that runs four times faster.

`render_card` writes the card from measured values only: every number in it comes from `meta`, there is no
default that could be mistaken for a measurement. A reference value the producer did not measure renders
`n/a`, and a latency that was not measured is written `non mesurée`, never `0`.

`check_card` reads a card back and returns what a reader should not have to trust: the phrases in
`FORBIDDEN`, and any speedup claim on a line that does not name both a device and a runtime — `2.3x faster`
says nothing, and so does `2.3x on CPU`; `2.3x on CPU with onnxruntime` says where it was measured.

In [ ]:
#| export
FORBIDDEN = ('lossless', 'un moteur int8', 'an int8 engine', 'state-of-the-art', 'sota', 'verified',
             'nan', 'ok=false', 'inconclusive', 'todo', 'tbd', 'xxx', 'placeholder')

_DEVICES = ('cpu', 'gpu', 'orin', '5090', 'jetson')
_RUNTIMES = ('tensorrt', 'onnxruntime', 'ort', 'openvino', 'pytorch', 'torchscript', 'eager')
_SPEEDUP = re.compile(r'\b\d+(?:\.\d+)?\s?[x×](?!\w)', re.I)
_HUB_ID = re.compile(r'^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$')   # what the Hub accepts in `base_model:`


def _accuracy(k, n):
    "k/n in percent, with its Wilson 95 % interval"
    lo, hi = wilson(k, n)
    return f"{k}/{n} = {100 * k / n:.2f} % (Wilson 95 % [{100 * lo:.2f}, {100 * hi:.2f}])"


def _is_count(v):
    "A measured count, never a bool passing itself off as one"
    return isinstance(v, int) and not isinstance(v, bool)


def _count(v):
    "A count the producer measured, or n/a"
    return f"{v:,}" if _is_count(v) else 'n/a'


def _gap(artifact, reference):
    "Change from the reference in percent; never a ratio, a smaller model is not a faster one"
    if not (_is_count(artifact) and _is_count(reference) and reference): return 'n/a'
    return f"{100 * (artifact - reference) / reference:+.1f} %"


def render_card(
    meta: dict,  # name, base_model, license (a string, or id and validated_by), datasets, tags, scope_line, input_shape, recipe, rows, latency, provenance, optional ladder, and reference: name, k, n, bytes, params, macs, peak_activation_bytes
) -> str:
    "Render the model card: front matter, scope, recipe, the four criteria against the reference, latency and provenance"
    ref, latency = meta['reference'], meta.get('latency')
    missing = [f for f in ('name', 'k', 'n') if f not in ref]
    if missing: raise KeyError(f"meta['reference'] has no {missing} — the card names what it compares to, and on how many images")
    lic = meta['license']
    lic_id = lic['id'] if isinstance(lic, dict) else lic
    validated = lic.get('validated_by') if isinstance(lic, dict) else lic   # a plain string is one a person chose
    base = meta['base_model']
    hub_id = _HUB_ID.match(base)
    out = (['---', 'library_name: fastermodels'] + ([f"license: {lic_id}"] if validated else [])
           + ([f"base_model: {base}"] if hub_id else []) + ['datasets:'])
    out += [f"  - {d}" for d in meta.get('datasets', [])]
    out += ['tags:'] + [f"  - {t}" for t in meta.get('tags', ['fasterai'])]
    out += ['---', '', f"# {meta['name']}", '', meta['scope_line'], '', '## Recipe', '']
    out += [f"- `{k}`: {v}" for k, v in (meta.get('recipe') or {}).items()]
    out += ['', '## Criteria', '',
            f"Top-1 on the evaluation set named above, size on disk, peak live activations and "
            f"multiply-accumulates — the last two for one image of {meta.get('input_shape', 'the evaluation resolution')} "
            f"at batch 1 — each against **{ref['name']}**.", '',
            "Top-1 intervals are Wilson 95 %; the gap is a paired bootstrap 95 % CI (2000 resamples, seed 0) "
            "with an exact McNemar p on the same images.", '']
    for r in meta.get('rows', []):
        out += [f"### {r['artifact']} (`{r['file']}`)", '',
                '| criterion | reference | this artifact | gap |', '|---|---|---|---|',
                f"| top-1 | {_accuracy(ref['k'], ref['n'])} | {_accuracy(r['k'], r['n'])} "
                f"| {r['delta']:+.2f} pt, 95 % CI [{r['lo']:+.2f}, {r['hi']:+.2f}], McNemar p {r['p_mcnemar']:.4f} |",
                f"| size | {_count(ref.get('bytes'))} B, {_count(ref.get('params'))} params "
                f"| {_count(r.get('bytes'))} B, {_count(r.get('params'))} params "
                f"| {_gap(r.get('bytes'), ref.get('bytes'))} bytes, {_gap(r.get('params'), ref.get('params'))} params |",
                f"| memory | {_count(ref.get('peak_activation_bytes'))} B | {_count(r.get('peak_activation_bytes'))} B "
                f"| {_gap(r.get('peak_activation_bytes'), ref.get('peak_activation_bytes'))} |",
                f"| MACs | {_count(ref.get('macs'))} | {_count(r.get('macs'))} | {_gap(r.get('macs'), ref.get('macs'))} |"]
        if r.get('target') is not None:   # a target the interval does not clear is not demonstrated, which is not a failure
            why = 'the interval straddles the target' if r['hi'] > r['target'] else 'the interval is entirely below the target'
            said = (f"met (lower bound {r['lo']:+.2f})" if r['lo'] > r['target']
                    else f"not demonstrated (lower bound {r['lo']:+.2f}; {why})")
            out += ['', f"Accuracy target: {r['target']:+.1f} pt — {said}"]
        out += ['']
    if meta.get('ladder'):
        out += ['## Variants', '', 'Other points on the same ladder, from the same source model:', '',
                '| variant | repo | top-1 gap (pt) | bytes | memory (B) | MACs |', '|---|---|---|---|---|---|']
        out += [f"| {v['name']} | `{v['repo']}` | {v['delta']:+.2f} | {_count(v.get('bytes'))} "
                f"| {_count(v.get('peak_activation_bytes'))} | {_count(v.get('macs'))} |" for v in meta['ladder']]
        out += ['']
    out += ['## Latency', '']
    if not latency: out += ['non mesurée']
    else:
        out += ['| device | runtime | precision | batch | median (ms) | runs |', '|---|---|---|---|---|---|']
        out += [f"| {r['device']} | {r['runtime']} | {r['precision']} | {r['batch']} | {r['median_ms']} | {r['n_runs']} |"
                for r in latency]
    out += (['', '## Provenance', ''] + ([] if hub_id else [f"- Source model: {base}"])
            + [f"- {k}: `{v}`" for k, v in (meta.get('provenance') or {}).items()])
    if not validated: out += [f"- License: {lic_id} (not yet validated by a person)"]
    return '\n'.join(out) + '\n'


def check_card(
    text: str,  # the card to read back
) -> list[str]:
    "Every forbidden phrase and every speedup claim without both a device and a runtime on its line; empty when the card is clean"
    found = [p for p in FORBIDDEN if re.search(rf"\b{re.escape(p)}\b", text, re.I)]
    for line in text.splitlines():
        claim, low = _SPEEDUP.search(line), line.lower()
        if claim and not (any(w in low for w in _DEVICES) and any(w in low for w in _RUNTIMES)):
            found.append(f"{claim.group().strip()} without both a device and a runtime on its line")
    return found

In [ ]:
show_doc(render_card)

In [ ]:
show_doc(check_card)

---

## Usage

```python
from fastermodels import render_card, check_card

meta = {
    'name': 'resnet18-pruned', 'base_model': 'torchvision/resnet18', 'license': 'bsd-3-clause',
    'datasets': ['frgfm/imagenette'], 'tags': ['fasterai', 'pruning'],
    'scope_line': 'Imagenette, n=3925, Wilson half-width about 1 pt; pipeline evidence, not a published claim.',
    'input_shape': '3x160x160',
    'recipe': {'prune': 'ratio 0.3, local, round_to 8', 'recovery': '3 epochs'},
    'reference': {'name': 'resnet18 fine-tuned on Imagenette', 'k': 3700, 'n': 3925,
                  'bytes': 44_726_568, 'params': 11_181_642, 'macs': 1_824_000_000,
                  'peak_activation_bytes': 3_211_264},
    'rows': [{'artifact': 'pruned FP32', 'file': 'model.safetensors', 'params': 8_900_000, 'bytes': 35_600_000,
              'macs': 1_368_000_000, 'peak_activation_bytes': 2_408_448,
              'k': 3680, 'n': 3925, 'delta': -0.51, 'lo': -1.2, 'hi': 0.2, 'p_mcnemar': 0.12}],
    'latency': None,
    'provenance': {'fasterai': '0.4.0', 'fastermodels': '0.1.0', 'torch': '2.9.1', 'measured_on': '2026-09-11'},
}

card = render_card(meta)
check_card(card)   # [] — nothing a reader has to take on trust
```

Every value in `rows` and in `reference` is measured by the producer: `k`/`n` and the paired delta come from
[the harness](01_eval.html), `params`, `macs` and `peak_activation_bytes` from `params`, `macs` and
`peak_activation_bytes`, and `bytes` is the size of the file on disk. The card is written to `README.md` in
the artifact directory, which is also what the Hub shows.

---

## See Also

- [Eval](01_eval.html) - where `k`, `n`, the delta and the agreement come from
- [Gate](03_gate.html) - condition 7 refuses to publish a card `check_card` flags
- [Model](00_model.html) - the artifact the card describes

Tests live in `nbs/tests/test_card.ipynb`.